In [ ]:
# pip install xgboost joblib
import pandas as pd
import numpy as np
import os
import gc
import time
import joblib # <--- NUEVA LIBRERÍA PARA GUARDAR EL MODELO
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
import xgboost as xgb
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    f1_score,
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    classification_report
)

def entrenar_evaluar_xgb(target_name):
    """
    - Descripción: Entrena, regulariza, optimiza y evalúa un modelo XGBoost.
                   Guarda el modelo óptimo en disco (.pkl) para uso posterior en SHAP.
    """
    dir_datos = "../../Datos/Datasets Finales"
    dir_resultados = "../../Resultados/Resultados (etapa 3 y 4)/XGBoost"
    os.makedirs(dir_resultados, exist_ok=True)

    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER'] 

    print("="*60)
    print(f"Iniciando entrenamiento y evaluación de XGBOOST para la variable objetivo: {target_name.upper()}")
    print("="*60) 

    # [1/5] y [2/5] Carga y Balanceo de datos
    print("[1/5] Cargando datasets de entrenamiento...")
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False)
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control.csv"), low_memory=False)

    print("[2/5] Generando muestra balanceada...")
    n_onco = len(df_onco_train)
    df_train_maestro = pd.concat([df_onco_train, df_control_train.sample(n=n_onco, random_state=42)], ignore_index=True)
    del df_onco_train, df_control_train; gc.collect()

    features = [col for col in df_train_maestro.columns if col not in cols_excluir]
    X_train = df_train_maestro[features]
    y_train = df_train_maestro[target_name]
    
    clases_unicas = np.unique(y_train)
    is_multiclass = len(clases_unicas) > 2

    print(f"      -> Dimensiones entrenamiento: {X_train.shape} | Clases: {len(clases_unicas)}")

    # [3/5] Configurar Grid Search Regularizado
    print("[3/5] Configurando Grid Search CV (K=5)...")
    cv_estrategia = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    if not is_multiclass:
        conteo_clases = y_train.value_counts()
        peso_clase_positiva = conteo_clases[0] / conteo_clases[1]
        xgb_base = xgb.XGBClassifier(
            tree_method='hist', scale_pos_weight=peso_clase_positiva,
            random_state=42, n_jobs=-1
        )
    else:
        xgb_base = xgb.XGBClassifier(tree_method='hist', random_state=42, n_jobs=-1)

    espacio_hiperparametros = {
        'learning_rate': [0.01, 0.1, 0.3], # Tasa de aprendizaje o magnitud de corrección de errores
        'max_depth': [3, 6, 10] # Profundidad máxima permitida para cada árbol individual
    }


    grid_search = GridSearchCV( # Instancia la validación cruzada orquestada
        estimator=xgb_base, # Usa el XGBoost base creado arriba
        param_grid=espacio_hiperparametros, # Inyecta las 9 configuraciones posibles
        cv=cv_estrategia, # Asigna los 5 pliegues asegurados mediante StratifiedKFold
        scoring='f1_macro', # Establece F1-Macro como métrica a maximizar
        n_jobs=1, # Evalúa los pliegues secuencialmente de a uno para no agotar la RAM
        verbose=3 # Activa el reporte detallado por pliegue en la consola
    )

    # 5. Entrenar y extraer métricas filtradas
    # Etapa 4: Ejecutar el proceso de entrenamiento y optimización con monitoreo de tiempo y filtrado de configuraciones inestables (std > 0.10)
    print("[4/5] Entrenando modelo y evaluando configuraciones...") # Anuncia inicio del entrenamiento
    inicio = time.time() # Registra el reloj del sistema al empezar
    grid_search.fit(X_train, y_train) # Desata el ajuste matemático de las combinaciones y pliegues
    fin = time.time() # Registra el reloj al terminar
    print(f"      -> Búsqueda completada en {round((fin - inicio)/60, 2)} minutos.") # Muestra el tiempo invertido total

    cv_resultados = pd.DataFrame(grid_search.cv_results_)
    ruta_cv = os.path.join(dir_resultados, f"Resultados_GridSearch_XGB_{target_name}.csv") # Genera la ruta del archivo de historial
    cv_resultados.to_csv(ruta_cv, index=False) # Guarda el reporte de todas las iteraciones en el disco duro
    print(f"      -> Evidencia de hiperparámetros guardada en: {ruta_cv}")

    config_estables = cv_resultados[cv_resultados['std_test_score'] <= 0.10]
    
    if config_estables.empty:
        print("      ADVERTENCIA: Todas las configuraciones tienen std > 0.10.") # Informa el problema
        print("      Se utilizará la de mayor promedio por defecto.") # Toma la decisión por defecto de Scikit-Learn
        mejor_modelo = grid_search.best_estimator_
    else: # Si hubieron configuraciones aprobadas (lo esperado)
        mejor_indice = config_estables['mean_test_score'].idxmax() # Busca en qué fila está el F1 promedio más alto de las estables
        mejores_params = config_estables.loc[mejor_indice, 'params'] # Saca el diccionario de hiperparámetros de esa fila
        mejor_f1 = config_estables.loc[mejor_indice, 'mean_test_score'] # Recupera el valor F1 numérico promedio
        mejor_std = config_estables.loc[mejor_indice, 'std_test_score'] # Recupera el valor de la desviación estándar
        
        print(f"      -> Mejor configuración estable encontrada:") # Anuncia éxito
        print(f"         Hiperparámetros: {mejores_params}") # Detalla cuáles parámetros ganaron
        print(f"         F1-Macro Promedio: {mejor_f1:.4f} (std: {mejor_std:.4f})") # Muestra su performance documentada

        mejor_modelo = clone(grid_search.estimator) # Crea una nueva red vacía de XGBoost manteniendo random_state y n_jobs
        mejor_modelo.set_params(**mejores_params) # Le asigna estrictamente los hiperparámetros que ganaron
        mejor_modelo.fit(X_train, y_train) # Lo entrena de forma definitiva con el 100% de la matriz de entrenamiento
        
    # GUARDADO DEL MODELO MAESTRO EN DISCO
    ruta_modelo = os.path.join(dir_resultados, f"Modelo_Optimo_XGBoost_{target_name}.pkl")
    joblib.dump(mejor_modelo, ruta_modelo)
    print(f"      -> Modelo óptimo guardado en: {ruta_modelo}")

    # MÉTRICAS DE ENTRENAMIENTO (Para demostrar que no hay sobreajuste)
    print("\n--- Rendimiento en entrenamiento: ---")
    y_pred_train = mejor_modelo.predict(X_train)
    if is_multiclass:
        print(f"F1-Score (Macro) Train: {f1_score(y_train, y_pred_train, average='macro'):.4f}")
    else:
        print(f"F1-Score (Clase 1) Train: {f1_score(y_train, y_pred_train, pos_label=1):.4f}")

    del df_train_maestro, X_train, y_train; gc.collect()

    # [5/5] Evaluación final en Prueba
    print("\n[5/5] Evaluando en conjunto de prueba...")
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control.csv"), low_memory=False)

    df_test_maestro = pd.concat([df_onco_test, df_control_test], ignore_index=True)
    X_test = df_test_maestro[features]
    y_test = df_test_maestro[target_name]
    total_instancias = len(y_test)

    y_pred = mejor_modelo.predict(X_test)
    y_pred_proba = mejor_modelo.predict_proba(X_test)

    print("\n--- Resultados finales de evaluación ---") 
    print(classification_report(y_test, y_pred))
    
    f1_macro_val = f1_score(y_test, y_pred, average='macro')
    
    if is_multiclass:
        y_test_bin = label_binarize(y_test, classes=clases_unicas)
        auc_roc_val = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
        auprc_val = average_precision_score(y_test_bin, y_pred_proba, average='weighted')
        brier_val = np.mean([brier_score_loss(y_test_bin[:, k], y_pred_proba[:, k]) for k in range(len(clases_unicas))])
        
        clases_temp, soportes_clases = np.unique(y_test, return_counts=True)
        tasa_base = sum([(soporte / total_instancias)**2 for soporte in soportes_clases])

        print(f"F1-Score (Macro): {f1_macro_val:.4f}")
        print(f"AUPRC (OvR Weighted): {auprc_val:.4f}")
        print(f"AUC-ROC (OvR Weighted): {auc_roc_val:.4f}")
        print(f"Brier Score (Multiclase): {brier_val:.4f}")
            
    else:
        f1_clase1_val = f1_score(y_test, y_pred, pos_label=1)
        auc_roc_val = roc_auc_score(y_test, y_pred_proba[:, 1])
        auprc_val = average_precision_score(y_test, y_pred_proba[:, 1])
        brier_val = brier_score_loss(y_test, y_pred_proba[:, 1])
        
        clases_temp, soportes_clases = np.unique(y_test, return_counts=True)
        indice_clase_1 = np.where(clases_temp == 1)[0][0]
        tasa_base = soportes_clases[indice_clase_1] / total_instancias
        
        print(f"F1-Score (Clase 1): {f1_clase1_val:.4f}")
        print(f"F1-Score (Macro): {f1_macro_val:.4f}")
        print(f"AUPRC: {auprc_val:.4f}")
        print(f"AUC-ROC: {auc_roc_val:.4f}")
        print(f"Brier Score: {brier_val:.4f}")

    # --- INICIO BLOQUE DE VALIDACIÓN DE LIFT AUPRC ---
    print("\n" + "-" * 60)
    print(f"Validación de Lift (en AUPRC): {target_name.upper()}")
    print("-" * 60)
    print(f"Total episodios de prueba: {total_instancias}")
    print(f"Tasa base (Prevalencia Azar): {tasa_base:.4f} ({tasa_base*100:.2f}%)")
    print(f"AUPRC obtenido por el modelo: {auprc_val:.4f}")
    
    umbral_minimo = tasa_base * 3.0
    lift_real = auprc_val / tasa_base
    
    print(f"Lift real logrado: {lift_real:.2f}x")
    
    if tasa_base < 0.15: # Condición estricta solo aplica si la clase minoritaria o prevalencia es menor al 15%
        print(f"AUPRC Mínimo exigido (Tasa Base x 3.0): {umbral_minimo:.4f}")
        if auprc_val > umbral_minimo:
            print("Resultado: Cumple condición de Lift > 3.0")
        else:
            print("Resultado: No cumple condición de Lift > 3.0")
    else:
        print("Resultado: Target suficientemente balanceado")
        
    # 7. Extraer Feature Importances (Reemplaza a los Odds Ratio en algoritmos de árbol)
    importancias = mejor_modelo.feature_importances_ # Toma del motor XGBoost el vector que mide la importancia relativa de variables
    
    df_importancias = pd.DataFrame({ # Crea una tabla para leer estos datos estructuradamente
        'Variable': features, # Empareja la importancia con el nombre del atributo original
        'Importancia_Relativa': importancias # Asigna el valor del peso interno
    }).sort_values(by='Importancia_Relativa', ascending=False) # Mueve las variables de mayor impacto al inicio
    
    df_importancias = df_importancias[df_importancias['Importancia_Relativa'] > 0] # Filtra eliminando predictores ignorados por el modelo
    
    ruta_imp = os.path.join(dir_resultados, f"XGB_Importancia_Predictores_{target_name}.csv") # Dinamiza el nombre de guardado del CSV
    df_importancias.to_csv(ruta_imp, index=False) # Escribe el CSV físicamente
    
    print(f"Importancias de variables guardadas en: {ruta_imp}") # Confirma guardado exitoso
    
    del df_test_maestro, X_test, y_test # Borra variables del Test de memoria RAM
    gc.collect() # Limpia la memoria final
    print("="*60, "\n")

In [8]:
entrenar_evaluar_xgb('MORTALIDAD')

Iniciando entrenamiento y evaluación de XGBOOST para la variable objetivo: MORTALIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones entrenamiento: (780416, 110) | Clases: 2
[3/5] Configurando Grid Search CV (K=5)...
[4/5] Entrenando modelo y evaluando configuraciones...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END ...learning_rate=0.01, max_depth=3;, score=0.536 total time=   5.8s
[CV 2/5] END ...learning_rate=0.01, max_depth=3;, score=0.537 total time=   4.4s
[CV 3/5] END ...learning_rate=0.01, max_depth=3;, score=0.537 total time=   3.6s
[CV 4/5] END ...learning_rate=0.01, max_depth=3;, score=0.538 total time=   3.7s
[CV 5/5] END ...learning_rate=0.01, max_depth=3;, score=0.536 total time=   3.6s
[CV 1/5] END ...learning_rate=0.01, max_depth=6;, score=0.564 total time=   5.5s
[CV 2/5] END ...learning_rate=0.01, max_depth=6;, score=0.564 total time=   5.3s
[CV 3/5] END ...learning_rate=0.01, max_depth

In [9]:
entrenar_evaluar_xgb('SEVERIDAD')

Iniciando entrenamiento y evaluación de XGBOOST para la variable objetivo: SEVERIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones entrenamiento: (780416, 110) | Clases: 4
[3/5] Configurando Grid Search CV (K=5)...
[4/5] Entrenando modelo y evaluando configuraciones...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END ...learning_rate=0.01, max_depth=3;, score=0.666 total time=  13.7s
[CV 2/5] END ...learning_rate=0.01, max_depth=3;, score=0.665 total time=  12.1s
[CV 3/5] END ...learning_rate=0.01, max_depth=3;, score=0.666 total time=  12.2s
[CV 4/5] END ...learning_rate=0.01, max_depth=3;, score=0.667 total time=  12.2s
[CV 5/5] END ...learning_rate=0.01, max_depth=3;, score=0.665 total time=  12.5s
[CV 1/5] END ...learning_rate=0.01, max_depth=6;, score=0.706 total time=  18.8s
[CV 2/5] END ...learning_rate=0.01, max_depth=6;, score=0.705 total time=  18.8s
[CV 3/5] END ...learning_rate=0.01, max_depth=

In [10]:
entrenar_evaluar_xgb('CONSUMO_RECURSOS')

Iniciando entrenamiento y evaluación de XGBOOST para la variable objetivo: CONSUMO_RECURSOS
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones entrenamiento: (780416, 110) | Clases: 3
[3/5] Configurando Grid Search CV (K=5)...
[4/5] Entrenando modelo y evaluando configuraciones...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END ...learning_rate=0.01, max_depth=3;, score=0.548 total time=  10.7s
[CV 2/5] END ...learning_rate=0.01, max_depth=3;, score=0.547 total time=   9.3s
[CV 3/5] END ...learning_rate=0.01, max_depth=3;, score=0.545 total time=   9.2s
[CV 4/5] END ...learning_rate=0.01, max_depth=3;, score=0.548 total time=   9.2s
[CV 5/5] END ...learning_rate=0.01, max_depth=3;, score=0.548 total time=  10.8s
[CV 1/5] END ...learning_rate=0.01, max_depth=6;, score=0.631 total time=  13.6s
[CV 2/5] END ...learning_rate=0.01, max_depth=6;, score=0.630 total time=  13.9s
[CV 3/5] END ...learning_rate=0.01, max